In [0]:
%sql
Create or replace connection if not exists datahub_earthquake_conn
type http
options(
  host = "https://earthquake.usgs.gov",
  port = 443,
  base_path = "/earthquakes/feed/v1.0",
  bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

conn = w.connections.get("datahub_earthquake_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"
base_url

In [0]:
url = f'{base_url}/summary/all_day.geojson'
url

In [0]:
dbutils.widgets.text('catalog_name','')
catalog_name = dbutils.widgets.get('catalog_name')
catalog_name

In [0]:
%py
spark.sql(f"use catalog {catalog_name}")
spark.sql(f"use schema bronze")

spark.sql("create volume if not exists earthquake_data")

In [0]:
import requests
import json
import datetime


response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Error getting the data from {url}")

data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)